# Guide 1 — Concepts

BattINFO is the implementation layer for the [EMMO domain-battery ontology](https://github.com/emmo-repo/domain-battery).
It gives you JSON schemas, a Python library, and a CLI for authoring, validating, and publishing battery metadata as machine-readable Linked Data.

This guide explains the data model, record types, and the semantic layer.

**Estimated time:** 10 minutes

In [ ]:
import os, json, warnings
from pathlib import Path

_here = Path.cwd()
if (_here / "src" / "battinfo").exists():
    REPO = _here
elif (_here.parent.parent / "src" / "battinfo").exists():
    REPO = _here.parent.parent
    os.chdir(REPO)
else:
    REPO = _here

print(f"Repo root: {REPO}")

## The data model

Every battery experiment generates four kinds of record. BattINFO formalises them as a provenance chain:

```
CellSpecification  ─────────────────────────────────
  │  "Panasonic NCR18650B is a cylindrical Li-ion cell, 3.4 Ah"
  │
  └─► CellInstance  ───────────────────────────────
        │  "This specific cell, serial number LAB-001"
        │
        └─► Test  ──────────────────────────────
              │  "1C cycle-life test at 25 °C on LAB-001"
              │
              └─► Dataset  ──────────────────
                    "The voltage/current time-series"
```

Each record has a permanent, opaque IRI under `https://w3id.org/battinfo/`. The chain is machine-readable — a downstream tool can follow links from a dataset all the way back to the cell specification.

## A real record

In [ ]:
# Load the canonical A123 cell-spec record that ships with the repo
a123 = json.loads(
    (REPO / "examples/cell-spec/A123__ANR26650M1-B.json").read_text(encoding="utf-8")
)

# Top-level structure
list(a123.keys())

In [ ]:
# The product identity block
a123["cell_spec"]

In [ ]:
# Quantitative specifications
a123["properties"]

## Record types

### CellSpecification — the specification

A cell spec is a **product specification**: datasheet-level information about a battery model, not a physical item.

In RDF it carries two parallel type assertions:
- **EMMO**: `BatteryCellSpecification` (subclass of `emmo:Description` — an information entity describing a class of cells)
- **schema.org**: `schema:CreativeWork` (the specification document as a creative artifact)

The physical EMMO classes for format and chemistry (`BatteryCell`, `CylindricalBattery`, `LithiumIonBattery`, ...) are placed on an `isDescriptionFor` anonymous node — not on the specification itself — because the specification is an information entity, not a physical cell.

### CellInstance — the physical item

A cell instance is a **specific physical cell** with a serial number. It always links to a cell spec.

In RDF it carries both `BatteryCell` (EMMO scientific type) and `schema:IndividualProduct` (schema.org type for a uniquely identifiable product item). The link back to the cell-spec record uses both `hasDescription` (EMMO) and `schema:isVariantOf` (schema.org) for full dual-vocabulary alignment.

### TestSpec and Test

A **test spec** is a reusable procedure definition. A **test** is one execution of a test spec on one cell instance. In RDF: `BatteryTest`.

### Dataset

A dataset links measured data files to the cell instance and test that produced them. In RDF: `schema:Dataset`.

## Identifiers and IRIs

In [ ]:
# Every BattINFO record has a stable, opaque IRI
iri = a123["cell_spec"]["id"]
print("IRI:", iri)

# The 16-character Crockford Base32 UID
uid = iri.split("/")[-1]
print("UID:", uid)

IRIs are:
- **Opaque** — carry no embedded meaning
- **Stable** — never change once minted
- **Deterministic** — the same record inputs always produce the same IRI

This makes them safe to cite in papers, reference in databases, and dereference over HTTP.

## The semantic layer — JSON-LD

In [ ]:
from battinfo import CellSpecification, publish

cell_spec = CellSpecification(
    manufacturer="A123 Systems",
    model="ANR26650M1-B",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="LFP",
    negative_electrode_basis="graphite",
    properties={
        "nominal_capacity": {"value": 2.5, "unit": "Ah"},
        "nominal_voltage":  {"value": 3.3, "unit": "V"},
    },
)

result = publish(cell_spec, destination="local", root=REPO / ".battinfo/notebooks/guide-01")
print("IRI:", result.canonical_iri)

In [ ]:
from battinfo.api import publish_record

output = publish_record(
    result.debug_paths["canonical_record_path"],
    target_root=REPO / ".battinfo/notebooks/guide-01-jsonld",
)
jsonld = json.loads(
    Path(output["output_dir"], "index.jsonld").read_text(encoding="utf-8")
)

# EMMO @type stacking from format + chemistry + electrode basis
print("@type:", jsonld["@type"])

In [ ]:
# Each quantitative spec becomes a ConventionalProperty node
print("Properties:")
for prop in jsonld.get("hasProperty", []):
    class_name = prop["@type"][0] if isinstance(prop["@type"], list) else prop["@type"]
    value = prop.get("hasNumericalPart", {}).get("hasNumericalValue")
    unit  = prop.get("hasMeasurementUnit", "").split("#")[-1]
    print(f"  {class_name}: {value}  [{unit}]")

### Exported JSON-LD file

`publish_record()` writes `index.jsonld`, `index.json`, and `index.html` to the resolver artifact directory. The JSON-LD file below is the one that would be served when a consumer dereferences the cell-spec IRI.

In [ ]:
jsonld_file = Path(output["output_dir"]) / "index.jsonld"
print(f"JSON-LD exported to: {jsonld_file.relative_to(REPO)}")
print()
print(json.dumps(jsonld, indent=2))

## Validation layers

| Layer | What it checks |
|---|---|
| JSON Schema | Field names, types, required fields, value constraints |
| Pydantic | Python-level type coercion and cross-field rules |
| JSON-LD | RDF parsability, URDNA2015 normalisability, no undefined type terms |

In [ ]:
from battinfo.validate.record import validate_record_report

record_path = Path(result.debug_paths["canonical_record_path"])
record_doc = json.loads(record_path.read_text(encoding="utf-8"))

report = validate_record_report(
    record_doc,
    source_root=REPO / ".battinfo/notebooks/guide-01",
)
print("ok:", report.ok, "| errors:", len(report.errors), "| warnings:", len(report.warnings))

## Next

- **[Guide 2 — First cell spec](02-first-cell-type.ipynb)**: full authoring workflow from materials to published record
- **[Guide 3 — Linked records](03-linked-records.ipynb)**: build the complete provenance chain and publish